<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week1_Day2_Lexical_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install rank_bm25 nltk biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.7 MB/s eta 0:00:00


In [18]:
import json
import os
from typing import List, Dict, Any
from Bio import Entrez
from rank_bm25 import BM25Okapi

# Set email globally for Entrez
Entrez.email = "sesimboyle@gmail.com"

def load_or_fetch_pubmed_data(
    filename: str = "pubmed_abstracts.json",
    query: str = "(single-cell OR spatial transcriptomics) AND (human brain) AND (novel cell type)",
    max_records: int = 10
) -> List[Dict[str, Any]]:
    """
    Loads PubMed abstracts from a local JSON file.
    If the file is missing or empty, fetches fresh data from the PubMed API.
    """
    documents = []

    # 1. Try to load existing data
    if os.path.exists(filename):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                documents = json.load(f)
            if len(documents) > 0:
                print(f"✅ Loaded {len(documents)} documents from '{filename}'.")
                return documents
        except Exception as e:
            print(f"⚠️ Error reading file: {e}. Will re-fetch.")

    # 2. Fetch from PubMed if needed
    print(f"🔄 Fetching data from PubMed for query: '{query}'")
    search_handle = Entrez.esearch(db="pubmed", term=query, retmax=max_records, sort="relevance")
    search_results = Entrez.read(search_handle)
    search_handle.close()

    pmids = search_results.get("IdList", [])
    if not pmids:
        print("❌ No PMIDs found. Try adjusting your search query.")
        return []

    fetch_handle = Entrez.efetch(db="pubmed", id=pmids, rettype="abstract", retmode="xml")
    records = Entrez.read(fetch_handle)
    fetch_handle.close()

    extracted_data = []
    articles_list = records.get('PubmedArticleSet', {}).get('PubmedArticle', [])

    # Handle edge case where only one article is returned (it's a dict, not a list)
    if isinstance(articles_list, dict):
        articles_list = [articles_list]

    for article in articles_list:
        try:
            pmid_node = article["MedlineCitation"]["PMID"]
            pmid = pmid_node.get("text", "") if isinstance(pmid_node, dict) else str(pmid_node)

            title_raw = article["MedlineCitation"]["Article"]["ArticleTitle"]
            title = title_raw.get("content", "No Title") if isinstance(title_raw, dict) else title_raw

            abstract_node = article["MedlineCitation"]["Article"].get("Abstract", {})
            abstract_text_raw = abstract_node.get("AbstractText", [])

            if isinstance(abstract_text_raw, str):
                abstract_text = abstract_text_raw
            elif isinstance(abstract_text_raw, list):
                abstract_parts = [part.get('content', '') if isinstance(part, dict) else str(part) for part in abstract_text_raw]
                abstract_text = " ".join(abstract_parts).strip()
            else:
                abstract_text = ""

            extracted_data.append({"pmid": pmid, "title": title, "abstract": abstract_text})
        except KeyError as e:
            continue # Skip malformed records

    if extracted_data:
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(extracted_data, f, indent=2)
        print(f"✅ Successfully fetched and saved {len(extracted_data)} documents!")

    return extracted_data


def build_bm25_index(documents: List[Dict[str, Any]]) -> BM25Okapi:
    """
    Tokenizes a list of document dictionaries and builds a BM25 search index.
    """
    print(f"📚 Building BM25 index with {len(documents)} documents...")
    tokenized_corpus = [doc["abstract"].lower().split() for doc in documents]
    return BM25Okapi(tokenized_corpus)


def search_bm25(bm25_index: BM25Okapi, documents: List[Dict[str, Any]], query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    Searches the BM25 index for a given query and returns the top_k ranked documents.
    """
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)

    # Zip scores with documents and sort descending by score
    ranked_results = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)

    return [{"score": score, "doc": doc} for score, doc in ranked_results[:top_k]]

In [19]:
# 1. Load or fetch the data
docs = load_or_fetch_pubmed_data()

# 2. Build the index (only if we have data)
if docs:
    bm25 = build_bm25_index(docs)

    # 3. Test the search
    test_query = "microglia alzheimer disease"
    print(f"\n🔍 Searching for: '{test_query}'\n")

    results = search_bm25(bm25, docs, test_query, top_k=5)

    print(f"{'RANK':<5} | {'SCORE':<8} | {'PMID':<10} | {'TITLE'}")
    print("-" * 90)
    for rank, result in enumerate(results, start=1):
        short_title = result["doc"]["title"][:60] + "..." if len(result["doc"]["title"]) > 60 else result["doc"]["title"]
        print(f"{rank:<5} | {result['score']:<8.4f} | {result['doc']['pmid']:<10} | {short_title}")
else:
    print("❌ No documents available to search.")

✅ Loaded 10 documents from 'pubmed_abstracts.json'.
📚 Building BM25 index with 10 documents...

🔍 Searching for: 'microglia alzheimer disease'

RANK  | SCORE    | PMID       | TITLE
------------------------------------------------------------------------------------------
1     | 1.9178   | 41456076   | Exploring cellular heterogeneity: single-cell and spatial tr...
2     | 1.1823   | 40585969   | Single-cell transcriptomics of vascularized human brain orga...
3     | 0.0000   | 34582785   | Single-nucleus transcriptome analysis reveals cell-type-spec...
4     | 0.0000   | 36544231   | Spatially resolved transcriptomics reveals genes associated ...
5     | 0.0000   | 41053013   | Mapping human brain cell type origin and diseases through si...
